## Configuration and Imports

In [51]:
import json
import re
import os

# ==========================================
# CONFIGURATION & GLOBAL SETTINGS
# ==========================================

# List the filenames you want to process
INPUT_FILES = [
    "../data/extraction_tests/3b/metamath_adapter_no_instruct_eval_3B.json",
    "../data/extraction_tests/snippets.json",
]

# Units and junk text to strip from extracted answers
UNITS_TO_IGNORE = [
    "feet",
    "minutes",
    "segments",
    "ounces",
    "cm3",
    "cm³",
    "units",
    "meters",
    "hours",
    "slices",
]

# Set to True for verbose output
VERBOSE = True

print("✅ Configuration loaded.")


✅ Configuration loaded.


## Extract EDA Logic

In [40]:
def detailed_extraction(item):
    raw_output = item.get("raw_output", "")
    if not raw_output:
        return None, "Empty"

    # 1. CLEANUP: Collapse loops and trailing junk
    clean_text = re.sub(r"(.{5,20}?)\1{3,}", r"\1", raw_output.strip())
    clean_text = re.sub(r"[\s\.\*]{5,}(?=$)", "", clean_text)

    # 2. DEFINE PATTERNS
    # FIX: Removed \s from math_pattern so it doesn't swallow "4\nTherefore..."
    math_pattern = r"\\frac\{[^{}]*\}\{[^{}]*\}|\\?\d+/\d+|\d+\.\d+|[\d\.]+\s*\\?π?|[\$\-]?\d+[a-zA-Z]?\(?[a-zA-Z0-9\+\-]*\)?|[\$\-]?\d+"

    def strip_prose(text):
        if not text:
            return text
        text = re.sub(r"[#\*]+", "", text).strip()
        math_matches = re.findall(math_pattern, text)
        if math_matches:
            # FIX: If it's a sentence inside a tag, take the last numeric-ish part
            return math_matches[-1].replace("$", "").strip()
        return text.strip(".").strip()

    # 3. HIGH PRIORITY: Explicit Tags
    tag_patterns = [
        (r"####\s*(.*)", "GSM8K Tag"),
        (
            r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}",
            "LaTeX Box",
        ),  # FIX: Improved recursive capture
    ]
    for pattern, label in tag_patterns:
        matches = re.findall(pattern, clean_text, re.DOTALL)
        if matches:
            res = strip_prose(matches[-1])
            if res:
                return res, label

    # 4. MEDIUM PRIORITY: Conclusion Sentences
    conclusion_markers = [
        (
            rf"(?:[Tt]herefore|[Ss]o|[Tt]hus|[Hh]ence|[Aa]nswer is):?\s*(?:there are|he spent|it is|she is)?\s*({math_pattern})",
            "Conclusion Sentence",
        ),
        (rf"(?:[Ff]inal [Aa]nswer):?\s*({math_pattern})", "Explicit Conclusion"),
    ]
    for pattern, label in conclusion_markers:
        matches = re.findall(pattern, clean_text[-300:], re.DOTALL)
        if matches:
            res = (
                matches[-1]
                if not isinstance(matches[-1], tuple)
                else next(m for m in matches[-1] if m)
            )
            return res.replace("$", "").strip().strip("."), label

    # 5. VARIABLE ASSIGNMENT: Bi-directional check
    # FIX: Updated to handle tuples correctly so it doesn't return "2 = x"
    var_pattern = rf"(?:[Xx]\s*=\s*({math_pattern}))|(({math_pattern})\s*=\s*[Xx])"
    var_matches = re.findall(var_pattern, clean_text)
    if var_matches:
        last_m = var_matches[-1]
        # Logic to pick the value part of the assignment
        res = (
            last_m[0] if last_m[0] else last_m[2] if len(last_m) > 2 else last_m[1]
        ).strip()
        if not re.search(r"[\*\/]", res):
            return res.replace("$", ""), "Variable Assignment"

    # 6. LOW PRIORITY: Narrative Search
    narrative_pattern = rf"(?:[Ii]s|has|equals):?\s*({math_pattern})(?!\s*[\+\-\*\/=])"
    narrative_matches = re.findall(narrative_pattern, clean_text[-200:])
    if narrative_matches:
        return narrative_matches[-1].replace("$", "").strip(), "Narrative Number"

    # 7. LAST RESORT: Absolute Numeric Fallback
    # This is the "Safety Net" you suggested
    backups = re.findall(math_pattern, clean_text)
    if backups:
        for b in reversed(backups):
            clean_b = b.replace("$", "").strip().rstrip(".")
            if len(clean_b) > 0 and not clean_b.isalpha():
                return clean_b, "Numeric Fallback"

    return None, "No Result Found"


In [41]:
def run_comprehensive_eval():
    for file_path in INPUT_FILES:
        if not os.path.exists(file_path):
            print(f"⚠️ File not found: {file_path}")
            continue

        with open(file_path, "r") as f:
            data = json.load(f)

        print(f"\n{'=' * 80}")
        print(f"REPORT FOR: {file_path}")
        print(
            f"{'ID':<4} | {'Status':<6} | {'Extracted':<15} | {'Truth':<10} | {'Method Used'}"
        )
        print(f"{'-' * 80}")

        stats = {"extracted": 0, "failed": 0, "correct": 0, "total": len(data)}
        failed_items = []  # List to store the actual objects that failed

        for item in data:
            val, method = detailed_extraction(item)
            gt = str(item.get("ground_truth", "")).strip()

            # Match Logic
            is_correct = False
            if val is not None:
                stats["extracted"] += 1
                if val == gt:
                    is_correct = True
                else:
                    # Fuzzy match fallback for numeric values
                    try:
                        v_norm = float(val.replace("π", "3.14"))
                        gt_norm = float(gt.replace("π", "3.14"))
                        if abs(v_norm - gt_norm) < 1e-4:
                            is_correct = True
                    except:
                        pass
            else:
                stats["failed"] += 1

            if is_correct:
                stats["correct"] += 1
            else:
                # Store the original item + the failed extraction attempt for debugging
                item_to_save = item.copy()
                item_to_save["extraction_attempted"] = (
                    val if val else "FAILED_TO_EXTRACT"
                )
                item_to_save["extraction_method"] = method
                failed_items.append(item_to_save)

            status_icon = "✅" if is_correct else "❌"
            display_val = val if val else "FAILED"

            print(
                f"{item.get('id', '?'):<4} | {status_icon:<6} | {display_val:<15} | {gt:<10} | {method}"
            )

        # --- EXPORT FAILURES ---
        if failed_items:
            # Generate a name like 'failed_snippets.json'
            base_name = os.path.basename(file_path)
            output_name = (
                f"../data/extraction_tests/test_evaluations/failed_{base_name}"
            )
            with open(output_name, "w") as out_f:
                json.dump(failed_items, out_f, indent=4)
            print(f"\n💾 Saved {len(failed_items)} failures to: {output_name}")

        # Final Summary
        print(f"{'-' * 80}")
        print(f"📊 SUMMARY:")
        print(f"   - Total Cases: {stats['total']}")
        print(
            f"   - Accuracy: {stats['correct']}/{stats['total']} ({(stats['correct'] / stats['total']) * 100:.1f}%)"
        )
        print(f"{'=' * 80}\n")


## Execution

In [52]:
if __name__ == "__main__":
    run_comprehensive_eval()



REPORT FOR: ../data/extraction_tests/3b/metamath_adapter_no_instruct_eval_3B.json
ID   | Status | Extracted       | Truth      | Method Used
--------------------------------------------------------------------------------
0    | ❌      | 2/3             | 6          | Numeric Fallback
1    | ❌      | 1.1             | 100        | Numeric Fallback
2    | ❌      | 2               | 1          | Numeric Fallback
3    | ✅      | 1.              | 1          | Narrative Number
4    | ❌      | 0               | 2          | Variable Assignment
5    | ❌      | 9               | 9y         | Numeric Fallback
6    | ❌      | 10              | 56         | Conclusion Sentence
7    | ❌      | 12              | 99         | Narrative Number
8    | ✅      | 93              | 93         | Numeric Fallback
9    | ❌      | 4.              | 400        | Narrative Number
10   | ❌      | 144             | 10         | Variable Assignment
11   | ❌      | 220.            | 10         | Variable Assignme